In [1]:
# 1. Installing libraries (gTTS path settings setup)
!pip install gtts tensorflow pillow requests

# 2. Importing and checking TensorFlow (Deep Learning Library)
import tensorflow as tf
import gtts
print("=== Setup Success ===")
print("TensorFlow Version:", tf.__version__)
print("gTTS Version:", gtts.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 4.0 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.4.2
    Uninstalling click-8.4.2:
      Successfully uninstalled click-8.4.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.25.1 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.
huggingface-hub 1.20.1 requires click>=8.4.0, but you have click 8.1.8 which is incompatible.
wandb 0.28.0 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.
=== Setup Success ===
TensorFlow Version: 2.20.0
gTTS Version: 2.5.4


In [2]:
import os

# Create folders for our classes
dataset_dir = "./sample_dataset"
classes = ["Tomato_Early_Blight", "Potato_Late_Blight", "Healthy"]

for c in classes:
    path = os.path.join(dataset_dir, c)
    os.makedirs(path, exist_ok=True)

print("=== Folders Created Successfully ===")
print("Created folders:", os.listdir(dataset_dir))

=== Folders Created Successfully ===
Created folders: ['Tomato_Early_Blight', 'Healthy', 'Potato_Late_Blight']


In [4]:
from PIL import Image
import numpy as np
import os

dataset_dir = "./sample_dataset"
classes = ["Tomato_Early_Blight", "Potato_Late_Blight", "Healthy"]

# Generate 10 dummy images for each class to train the model quickly
for c in classes:
    class_path = os.path.join(dataset_dir, c)
    for i in range(10):  # Creates 10 images per category
        # Creating a random array representing leaf pixels (size 224x224, 3 color channels)
        random_image_array = np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)
        img = Image.fromarray(random_image_array)
        img.save(os.path.join(class_path, f"leaf_{i}.jpg"))

print("=== Mock Leaf Images Created Successfully ===")
# Checking if images are there in Tomato folder
print("Images in Tomato_Early_Blight:", os.listdir(os.path.join(dataset_dir, "Tomato_Early_Blight")))

=== Mock Leaf Images Created Successfully ===
Images in Tomato_Early_Blight: ['leaf_5.jpg', 'leaf_8.jpg', 'leaf_4.jpg', 'leaf_0.jpg', 'leaf_7.jpg', 'leaf_9.jpg', 'leaf_6.jpg', 'leaf_2.jpg', 'leaf_3.jpg', 'leaf_1.jpg']


In [5]:
import tensorflow as tf
from tensorflow.keras import layers, models

print("Loading dataset into TensorFlow...")
# 1. Load the mock dataset from our folders
train_ds = tf.keras.utils.image_dataset_from_directory(
    "./sample_dataset",
    image_size=(224, 224),
    batch_size=4  # Small batch size for faster execution
)

print("\nInitializing MobileNetV2 Core...")
# 2. Setup Lightweight MobileNetV2 architecture
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights=None  # Using empty weights for fast mock training
)
base_model.trainable = False

# 3. Create our custom model pipeline
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(64, activation='relu'),
    layers.Dense(3, activation='softmax') # 3 outputs: Tomato, Potato, Healthy
])

# 4. Compile the Model
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\n=== Model Compiled Successfully ===")
# Show the structural blueprint of our AI model
model.summary()

Loading dataset into TensorFlow...
Found 30 files belonging to 3 classes.

Initializing MobileNetV2 Core...

=== Model Compiled Successfully ===


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        81,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,340,163 (8.93 MB)

 Trainable params: 82,179 (321.01 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [6]:
# 1. Start training the AI model with our mock images
print("Starting Model Training Sprints...")
history = model.fit(
    train_ds,
    epochs=5 # We are training for 5 rounds (epochs)
)

# 2. Save the final trained weights as an H5 file
model_file_name = "plant_disease_model.h5"
model.save(model_file_name)

print("\n=== AI Model Training Complete & Saved! ===")
print(f"Your model is saved as: {model_file_name}")

Starting Model Training Sprints...
Epoch 1/5
8/8 ━━━━━━━━━━━━━━━━━━━━ 4s 75ms/step - accuracy: 0.3333 - loss: 1.0991
Epoch 2/5
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 78ms/step - accuracy: 0.3333 - loss: 1.0987
Epoch 3/5
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 70ms/step - accuracy: 0.3333 - loss: 1.0987
Epoch 4/5
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 77ms/step - accuracy: 0.3333 - loss: 1.0987
Epoch 5/5
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 76ms/step - accuracy: 0.3333 - loss: 1.0988



=== AI Model Training Complete & Saved! ===
Your model is saved as: plant_disease_model.h5


In [7]:
import random
from gtts import gTTS
import os

# 1. Custom Logic function to calculate Plant Health Score
def calculate_plant_health(disease_confidence, humidity, temperature):
    # Base score starts with the inverse of the disease probability
    base_score = int((1.0 - disease_confidence) * 100)

    # Weather-based risk deduction
    # High humidity (>80%) increases fungal infection risk
    weather_risk = 0
    if humidity > 80:
        weather_risk += 15
    if temperature > 35 or temperature < 15:
        weather_risk += 10

    final_score = max(0, min(100, base_score - weather_risk))
    return final_score

# 2. Simulated Environment Inputs (Test parameters)
detected_disease = "Tomato Late Blight"
confidence = 0.85      # 85% confidence
current_humidity = 88   # High humidity (rainy climate simulation)
current_temp = 28      # 28 degrees Celsius

# Run the scoring engine
health_score = calculate_plant_health(confidence, current_humidity, current_temp)

# Determine the severity status based on score
if health_score > 80:
    status = "Healthy / Low Risk"
    tamil_advice = "பயிர் ஆரோக்கியமாக உள்ளது. இயற்கை உரங்களை தொடர்ந்து பயன்படுத்தவும்."
elif health_score >= 50:
    status = "Moderate Risk (மிதமான பாதிப்பு)"
    tamil_advice = "மிதமான இலை கருகல் நோய் அறிகுறி உள்ளது. வேப்ப எண்ணெய் கரைசலை தெளிக்கவும்."
else:
    status = "High Danger (அதிக பாதிப்பு)"
    tamil_advice = "பயிர் கடுமையாக பாதிக்கப்பட்டுள்ளது. பஞ்சகவ்யா கரைசலை உடனடியாக தெளிக்கவும்."

print("=== Plant Health Advisory Output ===")
print(f"Disease Detected: {detected_disease}")
print(f"Plant Health Score: {health_score}/100")
print(f"Risk Status: {status}")
print(f"Tamil Recommendation Text: {tamil_advice}\n")

# 3. Convert Tamil recommendation text to an audio voice note (.mp3)
print("Generating Tamil Voice Advisory...")
tts = gTTS(text=tamil_advice, lang='ta')
audio_file = "agri_voice_advisory.mp3"
tts.save(audio_file)

print(f"=== Audio Saved Successfully: {audio_file} ===")

=== Plant Health Advisory Output ===
Disease Detected: Tomato Late Blight
Plant Health Score: 0/100
Risk Status: High Danger (அதிக பாதிப்பு)
Tamil Recommendation Text: பயிர் கடுமையாக பாதிக்கப்பட்டுள்ளது. பஞ்சகவ்யா கரைசலை உடனடியாக தெளிக்கவும்.

Generating Tamil Voice Advisory...
=== Audio Saved Successfully: agri_voice_advisory.mp3 ===


In [8]:
from IPython.display import Audio

# Play the generated Tamil voice advisory audio directly
Audio("agri_voice_advisory.mp3", autoplay=True)

In [10]:
import tensorflow as tf
import numpy as np
from PIL import Image
import io
from IPython.display import display, Javascript
from google.colab.output import eval_js
from codecs import encode

# 1. Load the trained model we saved in Day 1
print("Loading our trained AI model...")
model = tf.keras.models.load_model('plant_disease_model.h5')

# 2. JavaScript code to access the Laptop Webcam directly in Colab
def take_photo(filename='photo.jpg', quality=0.8):
  js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Capture Photo';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      // Resize the output to fit the video element.
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      // Wait for Capture to be clicked.
      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
  display(js)
  data = eval_js('takePhoto({})'.format(quality))
  binary = b64decode(data.split(',')[1])
  with open(filename, 'wb') as f:
    f.write(binary)
  return filename

# We need b64decode to convert JavaScript image format to Python bytes
from base64 import b64decode

try:
  print("Starting Camera... Show a plant leaf (or just test with anything) to the camera!")
  filename = take_photo()
  print('Saved to {}'.format(filename))

  # 3. Preprocess the captured image for our MobileNetV2 model
  img = Image.open(filename).resize((224, 224))
  img_array = np.array(img) / 255.0  # Normalize to [0, 1]
  img_array = np.expand_dims(img_array, axis=0) # Make it (1, 224, 224, 3)

  # 4. Predict using our model
  prediction = model.predict(img_array)
  confidence = float(np.max(prediction))

  print("\n=== AI Camera Prediction Result ===")
  print(f"Confidence Level: {confidence * 100:.2f}%")

except Exception as err:
  print(str(err))

Loading our trained AI model...


Starting Camera... Show a plant leaf (or just test with anything) to the camera!


<IPython.core.display.Javascript object>

Saved to photo.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step

=== AI Camera Prediction Result ===
Confidence Level: 33.46%


In [11]:
import tensorflow as tf
import numpy as np
from PIL import Image
import io
import random
from gtts import gTTS
from IPython.display import display, Javascript, Audio
from google.colab.output import eval_js
from base64 import b64decode

# ==========================================
# STEP 1: Load the Trained AI Model (Day 1)
# ==========================================
print("🔄 Loading our trained AI model...")
model = tf.keras.models.load_model('plant_disease_model.h5')

# ==========================================
# STEP 2: Custom Plant Health Logic (Day 2)
# ==========================================
def calculate_plant_health(disease_confidence, humidity, temperature):
    base_score = int((1.0 - disease_confidence) * 100)
    weather_risk = 0
    if humidity > 80:
        weather_risk += 15
    if temperature > 35 or temperature < 15:
        weather_risk += 10
    final_score = max(0, min(100, base_score - weather_risk))
    return final_score

# ==========================================
# STEP 3: JavaScript Camera Setup (Day 3)
# ==========================================
def take_photo(filename='photo.jpg', quality=0.8):
  js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Capture Photo';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
  display(js)
  data = eval_js('takePhoto({})'.format(quality))
  binary = b64decode(data.split(',')[1])
  with open(filename, 'wb') as f:
    f.write(binary)
  return filename

# ==========================================
# STEP 4: Execution Engine (Integration!)
# ==========================================
try:
  print("\n📸 Starting Integrated System Camera... Show a leaf and click Capture!")
  filename = take_photo()
  print(f"✅ Photo Captured & Saved as: {filename}")

  # Image Preprocessing
  img = Image.open(filename).resize((224, 224))
  img_array = np.array(img) / 255.0
  img_array = np.expand_dims(img_array, axis=0)

  # Predict using AI Model
  prediction = model.predict(img_array)
  confidence = float(np.max(prediction))

  # Simulated Environment values for testing
  detected_disease = "Tomato Late Blight"
  current_humidity = 85
  current_temp = 29

  # Health Score Calculation
  health_score = calculate_plant_health(confidence, current_humidity, current_temp)

  # Determine Risk & Advisory text
  if health_score > 80:
    status = "Healthy / Low Risk"
    tamil_advice = "பயிர் ஆரோக்கியமாக உள்ளது. இயற்கை உரங்களை தொடர்ந்து பயன்படுத்தவும்."
  elif health_score >= 50:
    status = "Moderate Risk (மிதமான பாதிப்பு)"
    tamil_advice = "மிதமான இலை கருகல் நோய் அறிகுறி உள்ளது. வேப்ப எண்ணெய் கரைசலை தெளிக்கவும்."
  else:
    status = "High Danger (அதிக பாதிப்பு)"
    tamil_advice = "பயிர் கடுமையாக பாதிக்கப்பட்டுள்ளது. பஞ்சகவ்யா கரைசலை உடனடியாக தெளிக்கவும்."

  # Print consolidated outputs
  print("\n==============================================")
  print("      🍀 SYSTEM INTEGRATION OUTPUT 🍀        ")
  print("==============================================")
  print(f"📸 Image Processed: {filename}")
  print(f"🎯 Disease Detection Confidence: {confidence * 100:.2f}%")
  print(f"🌡️ Environment: Temp {current_temp}°C | Humidity {current_humidity}%")
  print(f"📊 Calculated Plant Health Score: {health_score}/100")
  print(f"⚠️ Risk Level: {status}")
  print(f"📢 Tamil Advice: {tamil_advice}")
  print("==============================================")

  # Generate & Play Audio
  print("\n🎙️ Synthesizing Tamil Voice Advisory...")
  tts = gTTS(text=tamil_advice, lang='ta')
  audio_file = "integrated_voice_advisory.mp3"
  tts.save(audio_file)
  print("🔊 Advisory Audio Ready! Playing below...")

  display(Audio(audio_file, autoplay=True))

except Exception as err:
  print(f"An error occurred: {str(err)}")

🔄 Loading our trained AI model...



📸 Starting Integrated System Camera... Show a leaf and click Capture!


<IPython.core.display.Javascript object>

✅ Photo Captured & Saved as: photo.jpg
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 811ms/step

      🍀 SYSTEM INTEGRATION OUTPUT 🍀        
📸 Image Processed: photo.jpg
🎯 Disease Detection Confidence: 33.46%
🌡️ Environment: Temp 29°C | Humidity 85%
📊 Calculated Plant Health Score: 51/100
⚠️ Risk Level: Moderate Risk (மிதமான பாதிப்பு)
📢 Tamil Advice: மிதமான இலை கருகல் நோய் அறிகுறி உள்ளது. வேப்ப எண்ணெய் கரைசலை தெளிக்கவும்.

🎙️ Synthesizing Tamil Voice Advisory...
🔊 Advisory Audio Ready! Playing below...


In [13]:
# ==========================================
# STEP 1: Install Gradio UI Library
# ==========================================
print("Installing Gradio Web UI library...")
!pip install -q gradio

import gradio as gr
import tensorflow as tf
import numpy as np
from PIL import Image
from gtts import gTTS
import os

# ==========================================
# STEP 2: Load our Trained AI Model
# ==========================================
print("🔄 Loading our trained AI model...")
model = tf.keras.models.load_model('plant_disease_model.h5')

# ==========================================
# STEP 3: Integrated Prediction Logic Function
# ==========================================
def predict_plant_health(input_image, humidity, temperature):
    if input_image is None:
        return "Please upload or capture an image first.", None

    # Preprocess the input image
    img = Image.fromarray(input_image).resize((224, 224))
    img_array = np.array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    # Model prediction
    prediction = model.predict(img_array)
    confidence = float(np.max(prediction))

    # Calculate Plant Health Score
    base_score = int((1.0 - confidence) * 100)
    weather_risk = 0
    if humidity > 80:
        weather_risk += 15
    if temperature > 35 or temperature < 15:
        weather_risk += 10
    health_score = max(0, min(100, base_score - weather_risk))

    # Risk Level & Advisory
    if health_score > 80:
        status = "Healthy / Low Risk (ஆரோக்கியமாக உள்ளது)"
        tamil_advice = "பயிர் ஆரோக்கியமாக உள்ளது. இயற்கை உரங்களை தொடர்ந்து பயன்படுத்தவும்."
    elif health_score >= 50:
        status = "Moderate Risk (மிதமான பாதிப்பு)"
        tamil_advice = "மிதமான இலை கருகல் நோய் அறிகுறி உள்ளது. வேப்ப எண்ணெய் கரைசலை தெளிக்கவும்."
    else:
        status = "High Danger (அதிக பாதிப்பு)"
        tamil_advice = "பயிர் கடுமையாக பாதிக்கப்பட்டுள்ளது. பஞ்சகவ்யா கரைசலை உடனடியாக தெளிக்கவும்."

    # Format Text Output
    output_text = f"""
    ======================================
         🍀 PLANT HEALTH DIAGNOSIS 🍀
    ======================================
    🎯 Disease Confidence : {confidence * 100:.2f}%
    🌡️ Humidity           : {humidity}%
    🔥 Temperature        : {temperature}°C
    📊 Plant Health Score : {health_score}/100
    ⚠️ Risk Status        : {status}
    📢 Tamil Advisory     : {tamil_advice}
    ======================================
    """

    # Generate Tamil TTS Audio
    tts = gTTS(text=tamil_advice, lang='ta')
    audio_file = "gradio_voice_advisory.mp3"
    tts.save(audio_file)

    return output_text, audio_file

# ==========================================
# STEP 4: Build & Launch the Gradio Web App
# ==========================================
print("🚀 Building and Launching Web Application Interface...")

app_interface = gr.Interface(
    fn=predict_plant_health,
    inputs=[
        gr.Image(label="Upload Leaf Photo or Use Webcam"),
        gr.Slider(minimum=0, maximum=100, value=85, label="Humidity (%)"),
        gr.Slider(minimum=0, maximum=50, value=29, label="Temperature (°C)")
    ],
    outputs=[
        gr.Textbox(label="Diagnosis & Health Summary", lines=10),
        gr.Audio(label="Tamil Voice Advisory Playback", type="filepath")
    ],
    title="🌱 Smart AI Plant Disease & Health Advisory Web App",
    description="Upload a leaf photo, adjust weather variables (Humidity/Temp), and get instant AI analysis along with a dynamic Tamil voice advisory!",
    theme="soft"
)

# Launch with a public shareable link
app_interface.launch(share=True, debug=True)

Installing Gradio Web UI library...
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 377, in run
    requirement_set = resolver.resolve(
                      ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/resolution/resolvelib/resolver.py", line 95, in resolve
    result = self._result = resolver.resolve(
                            ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_vendor/resolvelib/resolvers.py", line 546, in resolve
    state = resolution.resolve(requirements, max_rounds=max_rounds)
           

KeyboardInterrupt: 